# Krewson, Stephen. 2019. "Extracting Illustrated Pages from Digital Libraries with Python." _Programming Historian_. https://doi.org/10.46430/phen0084

## Setup and dependency installation

In [ ]:
%pip install internetarchive requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 5.9 MB/s eta 0:00:00


## Query Internet Archive

In [ ]:
import internetarchive as ia

query = "peter parley date:[1825 TO 1830] mediatype:texts"
vol_ids = [result['identifier'] for result in ia.search_items(query)]
# filter out the non-functional ids
vol_ids = [i for i in vol_ids if i.startswith('tales')]
vol_ids

## Download Image Archives

In [ ]:
import gzip
import os
import requests
import xml.etree.ElementTree as ET

from pathlib import Path

def ia_picture_download(item_id, out_dir: Path = None):
    """
    This function is lightly adapted from the code provided
    in the Programming Historian lesson.

    :param item_id: unique Internet Archive volume identifier
    :param out_dir: destination for images; if None, no download

    Note: if supplied, out_dir must be an existing directory and
    the caller must have write permissions in that directory

    :rtype list of pages with one or more blockType=Picture in Abbyy OCR data
    """

    print("[{}] Starting processing".format(item_id))

    # Use command-line client to see available metadata formats:
    # `ia metadata formats VOLUME_ID`

    # for this lesson, only the Abbyy file is needed
    returned_files = list(ia.get_files(item_id, formats=["Abbyy GZ"]))

    # make sure something got returned
    if len(returned_files) > 0:
        abbyy_file = returned_files[0].name
    else:
        print("[{}] Could not get Abbyy file".format(item_id))
        return None

    # download the abbyy file to CWD
    ia.download(item_id, formats=["Abbyy GZ"], ignore_existing=True, destdir=os.getcwd(), no_directory=True)

    # collect the pages with at least one picture block
    img_pages = []

    with gzip.open(abbyy_file) as fp:
        tree = ET.parse(fp)
        document = tree.getroot()
        for i, page in enumerate(document):
            for block in page:
                try:
                    if block.attrib['blockType'] == 'Picture':
                        img_pages.append(i)
                        break
                except KeyError:
                    continue

    # 0 is not a valid page for making GET requests to IA,
    #yet sometimes it's in the zipped Abbyy file
    img_pages = [page for page in img_pages if page > 0]

    # track for download progress report
    total_pages = len(img_pages)

    # OCR files are huge, so just delete once we have pagelist
    os.remove(abbyy_file)

    # if out_dir is not None, then also download page images
    if out_dir:

        # otherwise, create folder to put the images
        print("[{}] Making directory {}".format(item_id, out_dir))
        out_dir.mkdir(parents=True, exist_ok=True)

        # https://iiif.archivelab.org/iiif/documentation
        urls = [f"https://iiif.archive.org/iiif/{item_id}${page}/full/full/0/default.jpg" for page in img_pages]

        # no direct page download through API, DIY
        for i, page, url in zip(range(1,total_pages), img_pages, urls):
            print(url)
            outfile = out_dir / f"{str(page)}.jpg"

            if outfile.exists():
                print(f"{outfile} already exists.")

            rsp = requests.get(url, allow_redirects=True)
            if rsp.status_code == 200:
                print("[{}] Downloading page {} ({}/{})".format(item_id, page, i+1, total_pages))
                with open(outfile, "wb") as fp:
                    fp.write(rsp.content)
            else:
                raise rsp.status_code

    # return list of pages with 1+ picture blocks
    return img_pages

In [ ]:
from pathlib import Path

for item_id in vol_ids:
    out_dir = Path(os.getcwd()) / "items" / item_id

    ia_picture_download(item_id, out_dir)